## Import Libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType
from pyspark.sql.functions import col, when, lower, upper, trim, translate, to_timestamp, to_date, datediff, lit, length, count

## Reading from bronze layer

In [0]:
df = spark.table("olist.bronze.order_reviews")
df.display()

## Overview about the table

In [0]:
# Table info
print("=== Schema ===")
df.printSchema()

print("=== Row Count ===")
print(f"Total rows: {df.count()}")

print("=== Null Counts per Column ===")
df.select([
    F.count(F.when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()

## Transformations

### 1. Trim all whitespaces from string columns

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### 2. Normalize improperly represented nulls in string columns

In [0]:
NULL_STRINGS = ["", "null", "none", "n/a", "na", "unknown", "-"]

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(
            field.name,
            when(lower(trim(col(field.name))).isin(NULL_STRINGS), None)
            .otherwise(col(field.name))
        )

### 3. Normalize review text

In [0]:
df = df.withColumn("review_comment_title",   lower(col("review_comment_title"))) \
       .withColumn("review_comment_message", lower(col("review_comment_message")))

### 4. Cast columns to proper types

In [0]:
# review_score: string → int (use try_cast to handle malformed values)
# review_creation_date & review_answer_timestamp: string → timestamp
df = (
    df.withColumn("review_score", F.expr("try_cast(review_score as int)"))
      .withColumn(
          "review_creation_date",
          F.expr("try_to_timestamp(review_creation_date, 'yyyy-MM-dd HH:mm:ss')")
      )
      .withColumn(
          "review_answer_timestamp",
          F.expr("try_to_timestamp(review_answer_timestamp, 'yyyy-MM-dd HH:mm:ss')")
      )
)

### 5. Validate review_score

In [0]:
# review_score should be between 1 and 5 (inclusive)
df = df.withColumn(
    "has_valid_score",
    col("review_score").between(1, 5)
)

### 6. Derive useful analytical columns

In [0]:
# has_comment_title / has_comment_message: quick flags for UX/text analysis
# has_any_comment: true if either title or message is present
# review_response_days: how many days between creation and answer (service SLA)
# review_creation_date_only / year / month: partition-friendly helpers
df = (
    df.withColumn("has_comment_title",   col("review_comment_title").isNotNull())
      .withColumn("has_comment_message", col("review_comment_message").isNotNull())
      .withColumn(
          "has_any_comment",
          col("review_comment_title").isNotNull() | col("review_comment_message").isNotNull()
      )
      .withColumn(
          "review_response_days",
          datediff(col("review_answer_timestamp"), col("review_creation_date"))
      )
      .withColumn("review_creation_date_only", to_date(col("review_creation_date")))
      .withColumn("review_creation_year",      F.year(col("review_creation_date")))
      .withColumn("review_creation_month",     F.month(col("review_creation_date")))
)

### 7. Handle nulls

In [0]:
# Drop rows missing any identifying key or core business field
df = df.filter(
    col("review_id").isNotNull() &
    col("order_id").isNotNull() &
    col("review_score").isNotNull()
)

### 8. Remove duplicates

In [0]:
print(f"Rows before deduplication: {df.count()}")
print(f"Distinct review_id: {df.select('review_id').distinct().count()}")

df = df.dropDuplicates(['review_id'])

print(f"Rows after deduplication: {df.count()}")

## Quality Checks

In [0]:
print(f"Total rows after cleaning: {df.count()}")
print(f"Distinct review_id: {df.select('review_id').distinct().count()}")
print(f"Distinct order_id:  {df.select('order_id').distinct().count()}")
print(f"Rows with invalid score: {df.filter(~col('has_valid_score')).count()}")
print(f"Null review_score: {df.filter(col('review_score').isNull()).count()}")
print(f"Null review_creation_date: {df.filter(col('review_creation_date').isNull()).count()}")
print(f"Null review_answer_timestamp: {df.filter(col('review_answer_timestamp').isNull()).count()}")
print(f"Rows with any comment: {df.filter(col('has_any_comment')).count()}")

print("\n=== Review score distribution ===")
df.groupBy("review_score").count().orderBy("review_score").display()



df.display()

## Write to silver layer

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("olist.silver.order_reviews")